# جلسه ۷: ساخت عامل‌های LLM

## اهداف
- درک معماری عامل (حلقه مشاهده ← فکر ← عمل)
- ساخت یک عامل خودمختار که از ابزارها به صورت تکراری استفاده می‌کند
- پیاده‌سازی حافظه مکالمه
- مدیریت وظایف استدلال چند مرحله‌ای

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

**تفاوت کلیدی با جلسه ۶:** در فراخوانی تابع، LLM یک بار ابزار را فراخوانی می‌کند و پاسخ می‌دهد. یک **عامل** حلقه می‌زند — می‌تواند ابزارها را تا تکمیل کار بارها فراخوانی کند.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. عامل (Agent) چیست؟

یک **عامل** یک LLM است که می‌تواند:
- **مشاهده کند**: وضعیت فعلی و نتایج ابزار را بخواند
- **فکر کند**: تصمیم بگیرد قدم بعدی چیست
- **عمل کند**: ابزاری فراخوانی کند یا پاسخ نهایی بدهد
- **تکرار کند**: تا تکمیل وظیفه ادامه دهد

```
وظیفه کاربر → LLM فکر می‌کند → ابزار A فراخوانی → نتیجه → LLM دوباره فکر می‌کند → ابزار B فراخوانی → نتیجه → LLM پاسخ نهایی
```

این الگوی **ReAct** است: **استدلال** + **عمل** در یک حلقه.

## ۲. تعریف ابزارهای عامل

بیایید مجموعه‌ای از ابزارهای مفید به عامل خود بدهیم.

In [ ]:
import math

# پیاده‌سازی ابزارها
def calculator(expression):
    """ارزیابی ایمن یک عبارت ریاضی."""
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return json.dumps({"error": "Invalid characters in expression"})
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def search_knowledge(query):
    """شبیه‌سازی جستجو در پایگاه دانش."""
    knowledge = {
        "population": {"France": "68 million", "Japan": "125 million", "Brazil": "214 million", "Germany": "84 million"},
        "capital": {"France": "Paris", "Japan": "Tokyo", "Brazil": "Brasilia", "Germany": "Berlin"},
        "gdp": {"France": "$2.78 trillion", "Japan": "$4.23 trillion", "Brazil": "$1.92 trillion", "Germany": "$4.07 trillion"},
        "area_km2": {"France": 643801, "Japan": 377975, "Brazil": 8515767, "Germany": 357022}
    }
    results = []
    query_lower = query.lower()
    for category, data in knowledge.items():
        for country, value in data.items():
            if country.lower() in query_lower or category in query_lower:
                results.append({"country": country, "category": category, "value": value})
    return json.dumps(results if results else [{"message": "No results found"}])

def get_exchange_rate(from_currency, to_currency):
    """دریافت نرخ ارز (شبیه‌سازی شده)."""
    rates = {
        ("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09,
        ("USD", "JPY"): 149.50, ("JPY", "USD"): 0.0067,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
        ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16,
    }
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate:
        return json.dumps({"from": from_currency, "to": to_currency, "rate": rate})
    return json.dumps({"error": f"Rate not found for {from_currency} to {to_currency}"})

# اسکیمای ابزارها
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a mathematical expression. Supports +, -, *, /, parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search a knowledge base for country information (population, capital, GDP, area).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query about countries"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get the exchange rate between two currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_currency": {"type": "string", "description": "Source currency code (e.g., USD)"},
                    "to_currency": {"type": "string", "description": "Target currency code (e.g., EUR)"}
                },
                "required": ["from_currency", "to_currency"]
            }
        }
    }
]

available_functions = {
    "calculator": calculator,
    "search_knowledge": search_knowledge,
    "get_exchange_rate": get_exchange_rate
}

print(f"ابزارهای عامل: {list(available_functions.keys())}")

Agent tools: ['calculator', 'search_knowledge', 'get_exchange_rate']


## ۳. حلقه عامل

هسته اصلی عامل: فراخوانی مکرر LLM تا زمانی که پاسخ نهایی بدهد (بدون فراخوانی ابزار دیگر).

In [ ]:
def run_agent(user_message, max_iterations=5, verbose=True):
    """اجرای حلقه عامل که می‌تواند از ابزارها به صورت تکراری استفاده کند."""
    
    system_prompt = """You are a helpful research assistant with access to tools.
Use the available tools to find information and perform calculations.
Think step by step. Break complex questions into smaller parts.
When you have enough information, provide a clear final answer."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    for i in range(max_iterations):
        if verbose:
            print(f"\n--- مرحله عامل {i+1} ---")
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        message = response.choices[0].message
        
        # اگر فراخوانی ابزاری نبود، عامل کارش تمام شده
        if not message.tool_calls:
            if verbose:
                print("عامل به پایان رسید (فراخوانی ابزار دیگری نیست)")
            return message.content
        
        # پردازش فراخوانی‌های ابزار
        messages.append(message)
        
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            if verbose:
                print(f"  ابزار: {func_name}({func_args})")
            
            # اجرای تابع
            result = available_functions[func_name](**func_args)
            
            if verbose:
                print(f"  نتیجه: {result}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    
    return "عامل به حداکثر تعداد تکرار رسید بدون پاسخ نهایی."

print("تابع عامل تعریف شد!")

Agent function defined!


In [ ]:
# وظیفه ساده — نیاز به یک فراخوانی ابزار
result = run_agent("What is the population of France?")
print(f"\nپاسخ نهایی: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'population of France'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}]

--- Agent Step 2 ---


Agent finished (no more tool calls)

Final Answer: The population of France is about 68 million people.


In [ ]:
# وظیفه پیچیده — نیاز به چندین فراخوانی ابزار و استدلال
result = run_agent(
    "What is the population density of Japan? (population divided by area in km2)"
)
print(f"\nپاسخ نهایی: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'Japan population and area'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "Japan", "category": "capital", "value": "Tokyo"}, {"country": "Japan", "category": "gdp", "value": "$4.23 trillion"}, {"country": "Japan", "category": "area_km2", "value": 377975}]

--- Agent Step 2 ---


  Tool: calculator({'expression': '125000000 / 377975'})
  Result: {"expression": "125000000 / 377975", "result": 330.709703}

--- Agent Step 3 ---


Agent finished (no more tool calls)

Final Answer: Population: 125,000,000; area: 377,975 km\^2.

Calculation:
$$\text{density} = \frac{125000000}{377975} \approx 330.71\ \text{people/km}^2$$

Answer: ≈ 330.7 people per km².


In [ ]:
# حتی پیچیده‌تر — چند مرحله‌ای با ابزارهای مختلف
result = run_agent(
    "If France's GDP is divided equally among its population, how much would each person get in EUR?"
)
print(f"\nپاسخ نهایی: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'France GDP and population latest value GDP (nominal) and population'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "Japan", "category": "gdp", "value": "$4.23 trillion"}, {"country": "Brazil", "category": "gdp", "value": "$1.92 trillion"}, {"country": "Germany", "category": "gdp", "value": "$4.07 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}]

--- Agent Step 2 ---


  Tool: get_exchange_rate({'from_currency': 'USD', 'to_currency': 'EUR'})
  Result: {"from": "USD", "to": "EUR", "rate": 0.92}

--- Agent Step 3 ---


Agent finished (no more tool calls)

Final Answer: - Data used: GDP = \$2.78 trillion, population = 68 million, USD→EUR = 0.92.

Calculation (Python):
```python
# Python
gdp_usd = 2.78e12
pop = 68e6
rate = 0.92
per_person_eur = gdp_usd * rate / pop
per_person_eur
```

Math:
$$\text{per person}=\frac{2.78\times10^{12}\times0.92}{68\times10^{6}} \approx 37{,}609\ \text{EUR}.$$

Answer: about €37,609 per person.


## ۴. عامل با حافظه مکالمه

بیایید عاملی بسازیم که پیام‌های قبلی در مکالمه را به یاد می‌آورد.

In [ ]:
class ConversationalAgent:
    """عاملی با حافظه مکالمه."""
    
    def __init__(self, system_prompt=None):
        self.system_prompt = system_prompt or (
            "You are a helpful research assistant. Use tools when needed. "
            "Remember context from earlier in the conversation."
        )
        self.conversation_history = [
            {"role": "system", "content": self.system_prompt}
        ]
    
    def chat(self, user_message, max_iterations=5):
        """ارسال پیام و دریافت پاسخ، با حفظ تاریخچه."""
        self.conversation_history.append({"role": "user", "content": user_message})
        
        # ایجاد یک کپی کاری از پیام‌ها برای پردازش فراخوانی ابزار
        messages = self.conversation_history.copy()
        
        for _ in range(max_iterations):
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools
            )
            
            message = response.choices[0].message
            
            if not message.tool_calls:
                # افزودن پاسخ نهایی دستیار به تاریخچه
                self.conversation_history.append(
                    {"role": "assistant", "content": message.content}
                )
                return message.content
            
            # پردازش فراخوانی‌های ابزار
            messages.append(message)
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                result = available_functions[func_name](**func_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        return "به حداکثر تعداد تکرار رسید."

print("کلاس ConversationalAgent تعریف شد!")

ConversationalAgent defined!


In [ ]:
# تست حافظه مکالمه
agent = ConversationalAgent()

# نوبت ۱
print("کاربر: جمعیت آلمان چقدر است؟")
print(f"عامل: {agent.chat('What is the population of Germany?')}")

print()

# نوبت ۲ — به «آن کشور» از نوبت ۱ اشاره می‌کند
print("کاربر: تولید ناخالص داخلی آن کشور چقدر است؟")
print(f"عامل: {agent.chat('What about the GDP of that country?')}")

print()

# نوبت ۳ — به هر دو پاسخ قبلی اشاره می‌کند
print("کاربر: تولید ناخالص داخلی سرانه را محاسبه کن.")
print(f"عامل: {agent.chat('Now calculate the GDP per capita from those numbers.')}")

User: What's the population of Germany?


Agent: The population of Germany is about 84 million people.

User: What's the GDP of that country?


Agent: Germany's nominal GDP is about $4.07 trillion (approx.).

User: Calculate GDP per capita.


Agent: $$\text{GDP per capita}=\frac{4.07\times10^{12}}{84\times10^{6}}\approx 48{,}452\ \text{USD}$$


## تمرین: گسترش عامل

الگوی حلقه عامل برای هر ابزاری یکسان است.
سعی کنید سؤالات پیچیده چند مرحله‌ای بپرسید!

In [ ]:
# سؤالات پیچیده خودتان را امتحان کنید!
result = run_agent(
    "Compare the population density of France and Brazil. Which is more densely populated and by how much?"
)
print(f"\nپاسخ نهایی: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'population and area of France and Brazil population density France Brazil population area 2023'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "Brazil", "category": "capital", "value": "Brasilia"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "Brazil", "category": "gdp", "value": "$1.92 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}, {"country": "Brazil", "category": "area_km2", "value": 8515767}]

--- Agent Step 2 ---


  Tool: calculator({'expression': '68000000 / 643801'})
  Result: {"expression": "68000000 / 643801", "result": 105.6227}

--- Agent Step 3 ---


  Tool: calculator({'expression': '214000000 / 8515767'})
  Result: {"expression": "214000000 / 8515767", "result": 25.129856}

--- Agent Step 4 ---


  Tool: calculator({'expression': '105.6227 - 25.129856'})
  Result: {"expression": "105.6227 - 25.129856", "result": 80.492844}

--- Agent Step 5 ---


Agent finished (no more tool calls)

Final Answer: - France: population ≈ 68,000,000; area = 643,801 km² → density ≈ $ \\frac{68{,}000{,}000}{643{,}801} \\approx 105.6$ people/km²  
- Brazil: population ≈ 214,000,000; area = 8,515,767 km² → density ≈ $ \\frac{214{,}000{,}000}{8{,}515{,}767} \\approx 25.13$ people/km²

Conclusion: France is more densely populated by about $105.6-25.13\\approx 80.49$ people/km² (≈4.2× denser).


## خلاصه

**معماری عامل:**
1. دریافت وظیفه از کاربر
2. LLM تصمیم می‌گیرد از کدام ابزار استفاده کند (یا پاسخ نهایی بدهد)
3. اجرای ابزار، بازگرداندن نتیجه به LLM
4. تکرار تا تکمیل کار (یا رسیدن به حداکثر تکرار)

**نکات کلیدی:**
- عامل = LLM + ابزارها + حلقه
- همیشه `max_iterations` تنظیم کنید تا از حلقه‌های بی‌نهایت جلوگیری شود
- پرامپت سیستم خوب استدلال عامل را بهبود می‌بخشد
- حافظه مکالمه تعاملات چند نوبتی را ممکن می‌سازد
- وظایف پیچیده به طور خودکار توسط LLM به فراخوانی‌های ابزار تقسیم می‌شوند

**جلسه بعدی:** الگوهای پیشرفته — زنجیره‌سازی، محافظ‌ها، ارزیابی!